In [1]:
import pandas as pd
import numpy as np

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
mobility_spatial = pd.read_csv("mobility_store_proximity.csv")
opportunity_df = pd.read_csv("store_opportunity_analysis.csv")

print("Mobility spatial data:", mobility_spatial.shape)
print("Opportunity data:", opportunity_df.shape)

print("\nDatasets loaded successfully.")

Mobility spatial data: (5000, 22)
Opportunity data: (10, 16)

Datasets loaded successfully.


In [3]:
print("Mobility columns:")
print(mobility_spatial.columns.tolist())

print("\nOpportunity columns:")
print(opportunity_df.columns.tolist())

Mobility columns:
['ping_id', 'anonymous_user_id', 'timestamp', 'latitude', 'longitude', 'device_type', 'movement_type', 'date', 'hour', 'day_of_week', 'time_period', 'is_peak_hour', 'is_weekend', 'month', 'week_of_month', 'movement_type_code', 'nearest_store_id', 'nearest_store_name', 'distance_to_nearest_store_km', 'distance_band', 'within_500m', 'within_1km']

Opportunity columns:
['opportunity_rank', 'nearest_store_id', 'nearest_store_name', 'store_type', 'total_mobility_pings', 'avg_distance_km', 'pings_within_500m', 'pings_within_1km', 'catchment_500m_pct', 'catchment_1km_pct', 'mobility_intensity_score', 'catchment_score', 'capacity_pressure_score', 'nearby_store_count', 'cannibalization_indicator', 'opportunity_score']


In [4]:
store_time_analysis = (
    mobility_spatial
    .groupby(
        [
            "nearest_store_id",
            "nearest_store_name",
            "time_period"
        ]
    )
    .size()
    .reset_index(name="mobility_pings")
)

store_time_analysis

,nearest_store_id,nearest_store_name,time_period,mobility_pings
0,S001,GeoCafe Central,Afternoon,63
1,S001,GeoCafe Central,Evening,56
2,S001,GeoCafe Central,Morning,81
3,S001,GeoCafe Central,Night,116
4,S002,GeoCafe Market,Afternoon,45
5,S002,GeoCafe Market,Evening,35
6,S002,GeoCafe Market,Morning,60
7,S002,GeoCafe Market,Night,59
8,S003,GeoCafe Riverside,Afternoon,104
9,S003,GeoCafe Riverside,Evening,91


In [5]:
store_movement_analysis = (
    mobility_spatial
    .groupby(
        [
            "nearest_store_id",
            "nearest_store_name",
            "movement_type"
        ]
    )
    .size()
    .reset_index(name="mobility_pings")
    .sort_values(
        "mobility_pings",
        ascending=False
    )
)

store_movement_analysis

,nearest_store_id,nearest_store_name,movement_type,mobility_pings
29,S010,GeoCafe South,Walking,342
14,S005,GeoCafe Downtown,Walking,340
23,S008,GeoCafe TechPark,Walking,305
11,S004,GeoCafe Station,Walking,293
20,S007,GeoCafe University,Walking,278
8,S003,GeoCafe Riverside,Walking,254
26,S009,GeoCafe North,Walking,236
12,S005,GeoCafe Downtown,Driving,230
27,S010,GeoCafe South,Driving,200
21,S008,GeoCafe TechPark,Driving,199


In [6]:
weekday_analysis = (
    mobility_spatial
    .groupby(
        [
            "nearest_store_id",
            "nearest_store_name",
            "is_weekend"
        ]
    )
    .size()
    .reset_index(name="mobility_pings")
)

weekday_analysis["day_type"] = np.where(
    weekday_analysis["is_weekend"] == 1,
    "Weekend",
    "Weekday"
)

weekday_analysis

,nearest_store_id,nearest_store_name,is_weekend,mobility_pings,day_type
0,S001,GeoCafe Central,0,237,Weekday
1,S001,GeoCafe Central,1,79,Weekend
2,S002,GeoCafe Market,0,137,Weekday
3,S002,GeoCafe Market,1,62,Weekend
4,S003,GeoCafe Riverside,0,401,Weekday
5,S003,GeoCafe Riverside,1,135,Weekend
6,S004,GeoCafe Station,0,444,Weekday
7,S004,GeoCafe Station,1,158,Weekend
8,S005,GeoCafe Downtown,0,502,Weekday
9,S005,GeoCafe Downtown,1,200,Weekend


In [7]:
peak_hour_analysis = (
    mobility_spatial
    .groupby(
        [
            "nearest_store_id",
            "nearest_store_name",
            "is_peak_hour"
        ]
    )
    .size()
    .reset_index(name="mobility_pings")
)

peak_hour_analysis["peak_status"] = np.where(
    peak_hour_analysis["is_peak_hour"] == 1,
    "Peak Hour",
    "Non-Peak Hour"
)

peak_hour_analysis

,nearest_store_id,nearest_store_name,is_peak_hour,mobility_pings,peak_status
0,S001,GeoCafe Central,0,298,Non-Peak Hour
1,S001,GeoCafe Central,1,18,Peak Hour
2,S002,GeoCafe Market,0,189,Non-Peak Hour
3,S002,GeoCafe Market,1,10,Peak Hour
4,S003,GeoCafe Riverside,0,510,Non-Peak Hour
5,S003,GeoCafe Riverside,1,26,Peak Hour
6,S004,GeoCafe Station,0,584,Non-Peak Hour
7,S004,GeoCafe Station,1,18,Peak Hour
8,S005,GeoCafe Downtown,0,671,Non-Peak Hour
9,S005,GeoCafe Downtown,1,31,Peak Hour


In [8]:
dominant_time = (
    store_time_analysis
    .sort_values(
        ["nearest_store_id", "mobility_pings"],
        ascending=[True, False]
    )
    .drop_duplicates("nearest_store_id")
)

dominant_time = dominant_time[
    [
        "nearest_store_id",
        "nearest_store_name",
        "time_period",
        "mobility_pings"
    ]
].rename(
    columns={
        "time_period": "dominant_time_period",
        "mobility_pings": "dominant_period_pings"
    }
)

dominant_time

,nearest_store_id,nearest_store_name,dominant_time_period,dominant_period_pings
3,S001,GeoCafe Central,Night,116
6,S002,GeoCafe Market,Morning,60
11,S003,GeoCafe Riverside,Night,183
15,S004,GeoCafe Station,Night,204
19,S005,GeoCafe Downtown,Night,216
23,S006,GeoCafe Mall,Night,94
27,S007,GeoCafe University,Night,181
31,S008,GeoCafe TechPark,Night,205
35,S009,GeoCafe North,Night,176
39,S010,GeoCafe South,Night,228


In [9]:
total_store_pings = (
    mobility_spatial
    .groupby("nearest_store_id")
    .size()
    .reset_index(name="total_pings")
)

dominant_time = dominant_time.merge(
    total_store_pings,
    on="nearest_store_id",
    how="left"
)

dominant_time["dominant_period_pct"] = (
    dominant_time["dominant_period_pings"]
    / dominant_time["total_pings"]
    * 100
)

dominant_time["dominant_period_pct"] = (
    dominant_time["dominant_period_pct"].round(2)
)

dominant_time

,nearest_store_id,nearest_store_name,dominant_time_period,dominant_period_pings,total_pings,dominant_period_pct
0,S001,GeoCafe Central,Night,116,316,36.71
1,S002,GeoCafe Market,Morning,60,199,30.15
2,S003,GeoCafe Riverside,Night,183,536,34.14
3,S004,GeoCafe Station,Night,204,602,33.89
4,S005,GeoCafe Downtown,Night,216,702,30.77
5,S006,GeoCafe Mall,Night,94,283,33.22
6,S007,GeoCafe University,Night,181,517,35.01
7,S008,GeoCafe TechPark,Night,205,648,31.64
8,S009,GeoCafe North,Night,176,513,34.31
9,S010,GeoCafe South,Night,228,684,33.33


In [10]:
performance_df = opportunity_df.merge(
    dominant_time[
        [
            "nearest_store_id",
            "dominant_time_period",
            "dominant_period_pings",
            "dominant_period_pct"
        ]
    ],
    on="nearest_store_id",
    how="left"
)

performance_df

,opportunity_rank,nearest_store_id,nearest_store_name,store_type,total_mobility_pings,avg_distance_km,pings_within_500m,pings_within_1km,catchment_500m_pct,catchment_1km_pct,mobility_intensity_score,catchment_score,capacity_pressure_score,nearby_store_count,cannibalization_indicator,opportunity_score,dominant_time_period,dominant_period_pings,dominant_period_pct
0,1,S010,GeoCafe South,Residential,684,2.02,43,159,6.29,23.25,97.44,13.07,195.43,0,Low Overlap,100.00,Night,228,33.33
1,2,S007,GeoCafe University,University,517,1.24,91,299,17.60,57.83,73.65,33.69,172.33,0,Low Overlap,91.27,Night,181,35.01
2,3,S003,GeoCafe Riverside,Riverside,536,1.48,89,269,16.60,50.19,76.35,30.04,153.14,0,Low Overlap,85.49,Night,183,34.14
3,4,S005,GeoCafe Downtown,Downtown,702,1.85,76,242,10.83,34.47,100.00,20.29,127.64,0,Low Overlap,84.38,Night,216,30.77
4,5,S004,GeoCafe Station,Transit,602,1.91,68,193,11.30,32.06,85.75,19.60,120.40,0,Low Overlap,76.30,Night,204,33.89
5,6,S008,GeoCafe TechPark,Tech Park,648,1.74,68,235,10.49,36.27,92.31,20.80,108.00,0,Low Overlap,75.56,Night,205,31.64
6,7,S009,GeoCafe North,High Street,513,1.97,50,147,9.75,28.65,73.08,17.31,128.25,0,Low Overlap,72.90,Night,176,34.31
7,8,S001,GeoCafe Central,High Street,316,0.63,101,288,31.96,91.14,45.01,55.63,70.22,0,Low Overlap,55.76,Night,116,36.71
8,9,S002,GeoCafe Market,Market,199,0.65,64,174,32.16,87.44,28.35,54.27,49.75,0,Low Overlap,42.55,Morning,60,30.15
9,10,S006,GeoCafe Mall,Shopping Mall,283,1.17,69,162,24.38,57.24,40.31,37.52,40.43,0,Low Overlap,39.51,Night,94,33.22
